# EEG student review and segment preparation

This notebook provides a graphical workflow for reviewing BrainVision EEG recordings (`.vhdr`, `.eeg`, `.vmrk`). The original files are never modified.

Run Steps 1–5 in order:
1. Select one or more `.vhdr` files and an output folder in Windows dialogs.
2. Load the helper functions.
3. Choose one queued recording and create an untouched master plus a 0.5–50 Hz filtered copy.
4. Inspect all channels and annotations using the original stacked Plotly z-score display.
5. Use the GUI to standardize/insert annotations, rename channels, mark bad channels, confirm the selected range in Plotly, and save.

**Important:** Z-scoring is for visualization only. The saved reviewed master retains its physical amplitude units. Unannotated time is unreviewed, not automatically clean.


In [ ]:
%pip install h5py

In [ ]:
%pip install mne

In [1]:
# Setup: install missing packages if needed.
# The technical lead should run this once before the student begins.
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy",
    "plotly": "plotly",
    "scipy": "scipy",
    "mne": "mne",
    "PyQt6": "PyQt6",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("Installation complete. Restart the notebook kernel, then run this cell again.")
else:
    get_ipython().run_line_magic("gui", "qt5")
    print("GUI packages are ready.")


ERROR:root:
    Could not load requested Qt binding. Please ensure that
    PyQt4 >= 4.7, PyQt5, PyQt6, PySide >= 1.0.3, PySide2, or
    PySide6 is available, and only one is imported per session.

    Currently-imported Qt library:                              None
    PyQt5 available (requires QtCore, QtGui, QtSvg, QtWidgets): False
    PyQt6 available (requires QtCore, QtGui, QtSvg, QtWidgets): True
    PySide2 installed:                                          False
    PySide6 installed:                                          False
    Tried to load:                                              ['pyqt5']
    


GUI packages are ready.


In [2]:
# Step 1: select patient recordings and the review output folder.
# Running this cell opens normal Windows selection dialogs.

from pathlib import Path
from PyQt6.QtWidgets import QApplication, QFileDialog

_app = QApplication.instance() or QApplication([])
VHDR_FILES, _ = QFileDialog.getOpenFileNames(
    None,
    "Select one or more BrainVision EEG header files",
    "",
    "BrainVision header files (*.vhdr)",
)
if not VHDR_FILES:
    raise RuntimeError("No .vhdr files were selected.")

EXPORT_FOLDER = QFileDialog.getExistingDirectory(
    None,
    "Choose the folder where reviewed data will be saved",
)
if not EXPORT_FOLDER:
    raise RuntimeError("No output folder was selected.")

print(f"Selected {len(VHDR_FILES)} recording(s):")
for number, path in enumerate(VHDR_FILES, start=1):
    print(f"  {number}. {Path(path).name}")
print("Review output folder:", EXPORT_FOLDER)


Selected 2 recording(s):
  1. 20260730or.vhdr
  2. 20260730preop.vhdr
Review output folder: C:/Users/zhouz/OneDrive - UC Irvine/Documents/Anes_Liveamp8_Data/Reorganized_liveamp8_data


In [3]:
# Step 2: functions to load, plot, zoom, and export.

from pathlib import Path
import csv
import json
import math
import numpy as np
import plotly.graph_objects as go
from scipy.io import savemat


def _parse_key_value_file(path):
    sections = {}
    current = None
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith(";"):
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip().lower()
                sections[current] = {}
                continue
            if current is None or "=" not in line:
                continue
            key, value = line.split("=", 1)
            sections[current][key.strip()] = value.strip()
    return sections


def load_brainvision(vhdr_file):
    vhdr_path = Path(vhdr_file)
    folder = vhdr_path.parent
    header = _parse_key_value_file(vhdr_path)
    common = header.get("common infos", {})
    binary = header.get("binary infos", {})
    channels = header.get("channel infos", {})

    data_file = common["DataFile"]
    data_format = common.get("DataFormat", "").upper()
    orientation = common.get("DataOrientation", "").upper()
    n_channels = int(common["NumberOfChannels"])
    srate = 1_000_000 / float(common["SamplingInterval"])
    binary_format = binary.get("BinaryFormat", "").upper()

    if data_format != "BINARY":
        raise ValueError(f"This beginner notebook expects DataFormat=BINARY, found {data_format!r}.")
    if orientation != "MULTIPLEXED":
        raise ValueError(f"This notebook expects DataOrientation=MULTIPLEXED, found {orientation!r}.")

    dtype_map = {
        "IEEE_FLOAT_32": np.float32,
        "INT_16": np.int16,
        "UINT_16": np.uint16,
    }
    if binary_format not in dtype_map:
        raise ValueError(f"Unsupported BinaryFormat: {binary_format!r}")
    dtype = dtype_map[binary_format]

    labels = []
    scales = []
    for ch in range(1, n_channels + 1):
        entry = channels.get(f"Ch{ch}", f"Ch {ch},,1,")
        parts = entry.split(",")
        labels.append(parts[0].strip() or f"Ch {ch}")
        scale = float(parts[2]) if len(parts) >= 3 and parts[2].strip() else 1.0
        scales.append(scale)

    eeg_path = folder / data_file
    raw = np.fromfile(eeg_path, dtype=dtype)
    if raw.size % n_channels != 0:
        raise ValueError("The .eeg file length is not divisible by the number of channels.")
    n_samples = raw.size // n_channels
    data = raw.reshape(n_samples, n_channels).T.astype(float)
    data *= np.asarray(scales)[:, None]

    marker_path = folder / f"{vhdr_path.stem}.vmrk"
    markers = []
    if marker_path.exists():
        marker_info = _parse_key_value_file(marker_path).get("marker infos", {})
        for key, value in marker_info.items():
            parts = value.split(",")
            if len(parts) >= 3:
                marker_type = parts[0].strip()
                label = parts[1].strip().replace(r"\1", ",")
                sample = int(float(parts[2]))
                if marker_type.lower() == "comment" and label:
                    markers.append({"label": label, "sample": sample, "time_sec": (sample - 1) / srate})

    return {
        "vhdr_file": str(vhdr_path),
        "eeg_file": str(eeg_path),
        "marker_file": str(marker_path),
        "data": data,
        "labels": labels,
        "srate": srate,
        "n_samples": n_samples,
        "duration_sec": n_samples / srate,
        "markers": markers,
    }


def _downsample_indices(n_samples, max_points):
    step = max(1, math.ceil(n_samples / max_points))
    return np.arange(0, n_samples, step)


def _normalize_for_view(y, plot_mode):
    if plot_mode == "raw":
        return y, "Amplitude"

    if plot_mode == "zscore":
        center = np.nanmean(y, axis=1, keepdims=True)
        scale = np.nanstd(y, axis=1, keepdims=True)
        ylabel = "Z-score"
    elif plot_mode == "robust_zscore":
        center = np.nanmedian(y, axis=1, keepdims=True)
        mad = np.nanmedian(np.abs(y - center), axis=1, keepdims=True)
        scale = 1.4826 * mad
        ylabel = "Robust z-score"
    else:
        raise ValueError("plot_mode must be 'raw', 'zscore', or 'robust_zscore'.")

    scale[~np.isfinite(scale) | (scale == 0)] = 1.0
    return (y - center) / scale, ylabel


def plot_raw(eeg, start_sec=None, end_sec=None, channels="all", max_points=6000, plot_mode="zscore"):
    data = eeg["data"]
    labels = eeg["labels"]
    srate = eeg["srate"]
    if start_sec is None:
        start_sec = 0
    if end_sec is None:
        end_sec = eeg["duration_sec"]
    start_sample = max(0, int(round(start_sec * srate)))
    end_sample = min(eeg["n_samples"], int(round(end_sec * srate)))

    if channels == "all":
        channel_indices = list(range(data.shape[0]))
    else:
        channel_indices = [labels.index(ch) if isinstance(ch, str) else int(ch) for ch in channels]

    sample_indices = start_sample + _downsample_indices(end_sample - start_sample, max_points)
    t = sample_indices / srate / 60.0
    y_raw = data[np.ix_(channel_indices, sample_indices)]
    y, ylabel = _normalize_for_view(y_raw, plot_mode)

    robust = np.nanpercentile(np.abs(y - np.nanmedian(y)), 95)
    spacing = robust * 4 if np.isfinite(robust) and robust > 0 else 1.0
    offsets = np.arange(len(channel_indices))[::-1] * spacing

    fig = go.Figure()
    for row, ch in enumerate(channel_indices):
        fig.add_trace(go.Scattergl(
            x=t,
            y=y[row] + offsets[row],
            mode="lines",
            name=labels[ch],
            line={"width": 1},
        ))

    top = float(np.nanmax(y + offsets[:, None]))
    bottom = float(np.nanmin(y + offsets[:, None]))
    height = top - bottom if top > bottom else 1.0
    label_y = top + 0.05 * height

    for marker in eeg["markers"]:
        x = marker["time_sec"] / 60.0
        if start_sec <= marker["time_sec"] <= end_sec:
            fig.add_vline(x=x, line_color="red", line_dash="dash", opacity=0.65)
            fig.add_annotation(
                x=x,
                y=label_y,
                text=marker["label"],
                showarrow=False,
                textangle=-45,
                font={"color": "red", "size": 11},
                yanchor="bottom",
            )

    fig.update_layout(
        title=f"EEG with marker labels ({plot_mode})",
        xaxis_title="Time (minutes)",
        yaxis={"title": ylabel, "tickmode": "array", "tickvals": offsets, "ticktext": [labels[ch] for ch in channel_indices]},
        hovermode="x unified",
        height=max(520, 90 * len(channel_indices)),
        margin={"l": 80, "r": 30, "t": 100, "b": 50},
    )
    fig.update_yaxes(range=[bottom - 0.05 * height, top + 0.25 * height])
    fig.show()


def export_segment(eeg, start_sec, end_sec, segment_name, export_folder=EXPORT_FOLDER):
    export_path = Path(export_folder)
    export_path.mkdir(parents=True, exist_ok=True)

    srate = eeg["srate"]
    start_sample = max(0, int(round(start_sec * srate)))
    end_sample = min(eeg["n_samples"], int(round(end_sec * srate)))
    if end_sample <= start_sample:
        raise ValueError("End time must be after start time.")

    data_segment = eeg["data"][:, start_sample:end_sample]
    time_sec = np.arange(start_sample, end_sample) / srate
    segment_markers = [m for m in eeg["markers"] if start_sec <= m["time_sec"] <= end_sec]

    safe_name = "".join(c if c.isalnum() or c in "-_" else "_" for c in segment_name).strip("_")
    base = export_path / safe_name

    with open(base.with_suffix(".csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["time_sec", *eeg["labels"]])
        for i in range(data_segment.shape[1]):
            writer.writerow([time_sec[i], *data_segment[:, i]])

    np.savez_compressed(
        base.with_suffix(".npz"),
        data=data_segment,
        time_sec=time_sec,
        srate=srate,
        channel_labels=np.array(eeg["labels"], dtype=object),
        markers=np.array(segment_markers, dtype=object),
    )

    savemat(base.with_suffix(".mat"), {
        "data": data_segment,
        "time_sec": time_sec,
        "srate": srate,
        "channel_labels": np.array(eeg["labels"], dtype=object),
        "marker_labels": np.array([m["label"] for m in segment_markers], dtype=object),
        "marker_times_sec": np.array([m["time_sec"] for m in segment_markers]),
    })

    metadata = {
        "source_vhdr_file": eeg["vhdr_file"],
        "source_eeg_file": eeg["eeg_file"],
        "start_sec": start_sec,
        "end_sec": end_sec,
        "start_sample": start_sample,
        "end_sample_exclusive": end_sample,
        "srate": srate,
        "channel_labels": eeg["labels"],
        "markers_in_segment": segment_markers,
        "files": [str(base.with_suffix(ext)) for ext in [".csv", ".npz", ".mat", ".json"]],
    }
    with open(base.with_suffix(".json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print("Saved segment files:")
    print(base.with_suffix(".csv"))
    print(base.with_suffix(".npz"))
    print(base.with_suffix(".mat"))
    print(base.with_suffix(".json"))

# ---------- Beginner GUI review helpers ----------

import hashlib
import re
from datetime import datetime, timezone

import mne
from PyQt6.QtCore import Qt
from PyQt6.QtWidgets import (
    QApplication,
    QAbstractItemView,
    QCheckBox,
    QComboBox,
    QDialog,
    QDialogButtonBox,
    QDoubleSpinBox,
    QFileDialog,
    QFormLayout,
    QGroupBox,
    QHBoxLayout,
    QInputDialog,
    QLabel,
    QLineEdit,
    QListWidget,
    QMessageBox,
    QPushButton,
    QTableWidget,
    QTableWidgetItem,
    QVBoxLayout,
)


STANDARD_LABELS = [
    "phase_preop",
    "phase_induction",
    "phase_maintenance",
    "phase_emergence",
    "phase_postop",
    "propofol_drip",
    "sevoflurane",
    "propofol_bolus",
    "ketamine_bolus",
    "midazolam_bolus",
    "fentanyl_bolus",
    "morphine_bolus",
    "hydromorphone_bolus",
    "dexmedetomidine_bolus",
    "dexmedetomidine_drip",
    "lidocaine_bolus",
    "rocuronium_bolus",
    "Loss_of_responsiveness",
    "Return_of_responsiveness",
    "Airway_manipulation",
    "Incision",
    "BAD_eye_blink",
    "BAD_ECG",
    "BAD_movement",
    "BAD_electrocautery",
    "BAD_electrode_pop",
    "BAD_flat_channel",
    "BAD_line_noise",
    "BAD_other",
    "UNMAPPED_review_needed",
]


def _qt_app():
    """Return the existing Qt application, or create one."""
    return QApplication.instance() or QApplication([])


def choose_brainvision_files_and_output():
    """Open native Windows dialogs for source files and output folder."""
    _qt_app()
    files, _ = QFileDialog.getOpenFileNames(
        None,
        "Select one or more BrainVision EEG header files",
        "",
        "BrainVision header files (*.vhdr)",
    )
    if not files:
        raise RuntimeError("No .vhdr files were selected.")

    output = QFileDialog.getExistingDirectory(
        None,
        "Choose the folder where reviewed data will be saved",
    )
    if not output:
        raise RuntimeError("No output folder was selected.")
    return [str(Path(path)) for path in files], str(Path(output))


def _copy_annotations(annotations):
    return mne.Annotations(
        onset=annotations.onset.copy(),
        duration=annotations.duration.copy(),
        description=annotations.description.copy(),
        orig_time=annotations.orig_time,
        ch_names=list(annotations.ch_names),
    )


def prepare_gui_recording(vhdr_files):
    """Choose one queued file and create master, filtered, and display copies."""
    _qt_app()
    names = [Path(path).name for path in vhdr_files]
    chosen, accepted = QInputDialog.getItem(
        None,
        "Choose recording",
        "Recording to review:",
        names,
        0,
        False,
    )
    if not accepted:
        raise RuntimeError("Recording selection was cancelled.")

    source = Path(vhdr_files[names.index(chosen)])
    master = mne.io.read_raw_brainvision(source, preload=True, verbose="ERROR")
    if master.info["sfreq"] <= 100:
        raise ValueError(
            "A 50 Hz low-pass requires sampling frequency above 100 Hz."
        )

    eeg_picks = mne.pick_types(master.info, eeg=True, exclude=[])
    if len(eeg_picks) == 0:
        raise ValueError("No channels are typed as EEG.")

    filtered = master.copy().filter(
        l_freq=0.5,
        h_freq=50.0,
        picks=eeg_picks,
        method="fir",
        phase="zero",
        fir_design="firwin",
        skip_by_annotation=("edge", "bad_acq_skip"),
        verbose="ERROR",
    )

    display_raw = filtered.copy()
    display_data = display_raw.get_data()
    center = np.zeros(len(display_raw.ch_names), dtype=float)
    scale = np.ones(len(display_raw.ch_names), dtype=float)
    for pick in eeg_picks:
        channel = display_data[pick]
        center[pick] = np.nanmean(channel)
        channel_scale = np.nanstd(channel)
        scale[pick] = channel_scale if np.isfinite(channel_scale) and channel_scale > 0 else 1.0
        # MNE stores EEG in volts. In this display copy, 1 displayed microvolt
        # represents 1 z-score unit. The master data remain unchanged.
        display_data[pick] = ((channel - center[pick]) / scale[pick]) * 1e-6
    display_raw._data = display_data

    return {
        "source": source,
        "master": master,
        "filtered": filtered,
        "display": display_raw,
        "eeg_picks": list(eeg_picks),
        "zscore_center_volts": center,
        "zscore_scale_volts": scale,
        "original_channel_names": list(master.ch_names),
        "label_audit": [],
        "channel_audit": [],
    }


def _remove_vocab_placeholders(raw, placeholders):
    keep = []
    for index, (onset, duration, description) in enumerate(
        zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description)
    ):
        is_placeholder = (
            description in placeholders
            and abs(float(onset)) < 1e-12
            and abs(float(duration)) < 1e-12
        )
        if not is_placeholder:
            keep.append(index)
    raw.set_annotations(raw.annotations[keep] if keep else mne.Annotations([], [], []))


def sync_display_review_to_master(review):
    """Copy annotations and globally bad channels from display to all copies."""
    display_raw = review["display"]
    master = review["master"]
    filtered = review["filtered"]

    bads = list(display_raw.info["bads"])
    annotations = _copy_annotations(display_raw.annotations)
    for raw in (master, filtered):
        raw.info["bads"] = bads
        raw.set_annotations(_copy_annotations(annotations))


def open_interactive_review_viewer(review, start_sec=0.0, duration_sec=30.0):
    """Open the MNE Qt viewer and synchronize edits when it closes."""
    display_raw = review["display"]
    existing = set(str(value) for value in display_raw.annotations.description)
    placeholders = [label for label in STANDARD_LABELS if label not in existing]
    for label in placeholders:
        display_raw.annotations.append(0.0, 0.0, label)

    mne.viz.set_browser_backend("qt")
    display_raw.plot(
        start=max(0.0, float(start_sec)),
        duration=max(1.0, float(duration_sec)),
        n_channels=len(display_raw.ch_names),
        scalings={"eeg": 4e-6},
        remove_dc=False,
        block=True,
        show=True,
        title=(
            "0.5–50 Hz standardized EEG: 1 displayed µV = 1 z unit | "
            "Click channel=bad | Press A=annotations"
        ),
    )

    _remove_vocab_placeholders(display_raw, set(placeholders))
    sync_display_review_to_master(review)


class ReviewMetadataDialog(QDialog):
    """GUI for standardized labels, channel names, and annotation insertion."""

    def __init__(self, review, parent=None):
        super().__init__(parent)
        self.review = review
        self.raw = review["master"]
        self.setWindowTitle("Standardize EEG metadata and annotations")
        self.resize(1150, 800)

        instructions = QLabel(
            "Rename channels in the table. For every annotation, choose a "
            "standard label or keep the original. Check Delete only for an "
            "incorrect annotation. New annotations may apply to all channels "
            "or selected channels."
        )
        instructions.setWordWrap(True)

        channels_box = QGroupBox("Channel labels and globally bad channels")
        channels_layout = QVBoxLayout(channels_box)
        self.channel_table = QTableWidget(len(self.raw.ch_names), 4)
        self.channel_table.setHorizontalHeaderLabels(
            ["Original channel", "New channel label", "Globally bad", "Reason"]
        )
        for row, channel in enumerate(self.raw.ch_names):
            original = QTableWidgetItem(channel)
            original.setFlags(original.flags() & ~Qt.ItemFlag.ItemIsEditable)
            self.channel_table.setItem(row, 0, original)
            self.channel_table.setItem(row, 1, QTableWidgetItem(channel))
            bad = QCheckBox()
            bad.setChecked(channel in self.raw.info["bads"])
            self.channel_table.setCellWidget(row, 2, bad)
            self.channel_table.setItem(row, 3, QTableWidgetItem(""))
        self.channel_table.resizeColumnsToContents()
        channels_layout.addWidget(self.channel_table)

        annotations_box = QGroupBox("Existing annotations")
        annotations_layout = QVBoxLayout(annotations_box)
        self.annotation_table = QTableWidget(len(self.raw.annotations), 7)
        self.annotation_table.setHorizontalHeaderLabels(
            [
                "Onset (sec)",
                "Duration (sec)",
                "Original label",
                "Standard label",
                "Channels",
                "Delete",
                "Reviewer note",
            ]
        )
        for row, (onset, duration, description, channels) in enumerate(
            zip(
                self.raw.annotations.onset,
                self.raw.annotations.duration,
                self.raw.annotations.description,
                self.raw.annotations.ch_names,
            )
        ):
            for column, value in enumerate(
                [
                    f"{float(onset):.3f}",
                    f"{float(duration):.3f}",
                    str(description),
                ]
            ):
                item = QTableWidgetItem(value)
                item.setFlags(item.flags() & ~Qt.ItemFlag.ItemIsEditable)
                self.annotation_table.setItem(row, column, item)
            label = QComboBox()
            label.addItems(["KEEP_ORIGINAL"] + STANDARD_LABELS)
            if str(description) in STANDARD_LABELS:
                label.setCurrentText(str(description))
            self.annotation_table.setCellWidget(row, 3, label)
            channel_text = ",".join(channels) if channels else "all"
            channel_item = QTableWidgetItem(channel_text)
            channel_item.setFlags(channel_item.flags() & ~Qt.ItemFlag.ItemIsEditable)
            self.annotation_table.setItem(row, 4, channel_item)
            self.annotation_table.setCellWidget(row, 5, QCheckBox())
            self.annotation_table.setItem(row, 6, QTableWidgetItem(""))
        self.annotation_table.resizeColumnsToContents()
        annotations_layout.addWidget(self.annotation_table)

        add_box = QGroupBox("Insert a new annotation")
        add_layout = QFormLayout(add_box)
        self.add_onset = QDoubleSpinBox()
        self.add_onset.setRange(0.0, float(self.raw.times[-1]))
        self.add_onset.setDecimals(3)
        self.add_duration = QDoubleSpinBox()
        self.add_duration.setRange(0.001, float(self.raw.times[-1]))
        self.add_duration.setDecimals(3)
        self.add_duration.setValue(1.0)
        self.add_label = QComboBox()
        self.add_label.addItems(STANDARD_LABELS)
        self.add_channels = QListWidget()
        self.add_channels.setSelectionMode(
            QAbstractItemView.SelectionMode.ExtendedSelection
        )
        self.add_channels.addItem("ALL")
        self.add_channels.addItems(self.raw.ch_names)
        self.add_channels.item(0).setSelected(True)
        self.add_button = QPushButton("Add annotation to table")
        self.add_button.clicked.connect(self._add_annotation_row)
        add_layout.addRow("Onset:", self.add_onset)
        add_layout.addRow("Duration:", self.add_duration)
        add_layout.addRow("Standard label:", self.add_label)
        add_layout.addRow("Affected channels:", self.add_channels)
        add_layout.addRow("", self.add_button)

        zoom_box = QGroupBox("Optional range to reopen in the EEG viewer")
        zoom_layout = QFormLayout(zoom_box)
        self.zoom_start = QDoubleSpinBox()
        self.zoom_start.setRange(0.0, float(self.raw.times[-1]))
        self.zoom_start.setDecimals(3)
        self.zoom_end = QDoubleSpinBox()
        self.zoom_end.setRange(0.001, float(self.raw.times[-1]))
        self.zoom_end.setDecimals(3)
        self.zoom_end.setValue(min(120.0, float(self.raw.times[-1])))
        zoom_layout.addRow("Start:", self.zoom_start)
        zoom_layout.addRow("End:", self.zoom_end)

        buttons = QDialogButtonBox(
            QDialogButtonBox.StandardButton.Apply
            | QDialogButtonBox.StandardButton.Cancel
        )
        buttons.button(QDialogButtonBox.StandardButton.Apply).clicked.connect(
            self._validate_and_accept
        )
        buttons.rejected.connect(self.reject)

        bottom = QHBoxLayout()
        bottom.addWidget(add_box, 2)
        bottom.addWidget(zoom_box, 1)

        layout = QVBoxLayout(self)
        layout.addWidget(instructions)
        layout.addWidget(channels_box, 1)
        layout.addWidget(annotations_box, 2)
        layout.addLayout(bottom)
        layout.addWidget(buttons)

    def _add_annotation_row(self):
        selected = [item.text() for item in self.add_channels.selectedItems()]
        if not selected:
            QMessageBox.warning(self, "Missing channels", "Select ALL or channels.")
            return
        onset = float(self.add_onset.value())
        duration = float(self.add_duration.value())
        if onset + duration > self.raw.times[-1] + 1 / self.raw.info["sfreq"]:
            QMessageBox.warning(
                self, "Invalid interval", "The interval extends past the recording."
            )
            return

        row = self.annotation_table.rowCount()
        self.annotation_table.insertRow(row)
        values = [
            f"{onset:.3f}",
            f"{duration:.3f}",
            "(new annotation)",
        ]
        for column, value in enumerate(values):
            item = QTableWidgetItem(value)
            item.setFlags(item.flags() & ~Qt.ItemFlag.ItemIsEditable)
            self.annotation_table.setItem(row, column, item)
        label = QComboBox()
        label.addItems(STANDARD_LABELS)
        label.setCurrentText(self.add_label.currentText())
        self.annotation_table.setCellWidget(row, 3, label)
        channel_text = "all" if "ALL" in selected else ",".join(selected)
        channel_item = QTableWidgetItem(channel_text)
        channel_item.setFlags(channel_item.flags() & ~Qt.ItemFlag.ItemIsEditable)
        self.annotation_table.setItem(row, 4, channel_item)
        self.annotation_table.setCellWidget(row, 5, QCheckBox())
        self.annotation_table.setItem(row, 6, QTableWidgetItem("manually inserted"))

    def _validate_and_accept(self):
        new_names = [
            self.channel_table.item(row, 1).text().strip()
            for row in range(self.channel_table.rowCount())
        ]
        if any(not value for value in new_names):
            QMessageBox.warning(self, "Invalid channel label", "Channel labels cannot be blank.")
            return
        if len(set(new_names)) != len(new_names):
            QMessageBox.warning(self, "Duplicate channel label", "Channel labels must be unique.")
            return
        if self.zoom_end.value() <= self.zoom_start.value():
            QMessageBox.warning(self, "Invalid range", "Zoom end must be after zoom start.")
            return
        self.accept()

    def apply_changes(self):
        old_names = list(self.raw.ch_names)
        new_names = [
            self.channel_table.item(row, 1).text().strip()
            for row in range(self.channel_table.rowCount())
        ]
        rename_map = {
            old: new for old, new in zip(old_names, new_names) if old != new
        }
        bad_old_names = [
            old_names[row]
            for row in range(len(old_names))
            if self.channel_table.cellWidget(row, 2).isChecked()
        ]
        reasons = {
            old_names[row]: self.channel_table.item(row, 3).text().strip()
            for row in range(len(old_names))
            if self.channel_table.item(row, 3).text().strip()
        }

        onset = []
        duration = []
        description = []
        ch_names = []
        label_audit = []
        for row in range(self.annotation_table.rowCount()):
            if self.annotation_table.cellWidget(row, 5).isChecked():
                continue
            row_onset = float(self.annotation_table.item(row, 0).text())
            row_duration = float(self.annotation_table.item(row, 1).text())
            raw_label = self.annotation_table.item(row, 2).text()
            selected_label = self.annotation_table.cellWidget(row, 3).currentText()
            standard_label = (
                raw_label if selected_label == "KEEP_ORIGINAL" else selected_label
            )
            channel_text = self.annotation_table.item(row, 4).text()
            old_channels = [] if channel_text == "all" else channel_text.split(",")
            mapped_channels = [rename_map.get(value, value) for value in old_channels]
            note = self.annotation_table.item(row, 6).text().strip()

            onset.append(row_onset)
            duration.append(row_duration)
            description.append(standard_label)
            ch_names.append(mapped_channels)
            label_audit.append(
                {
                    "onset_sec": row_onset,
                    "duration_sec": row_duration,
                    "raw_label": raw_label,
                    "standard_label": standard_label,
                    "reviewer_note": note,
                }
            )

        for raw in (
            self.review["master"],
            self.review["filtered"],
            self.review["display"],
        ):
            if rename_map:
                raw.rename_channels(rename_map)
            raw.info["bads"] = [rename_map.get(name, name) for name in bad_old_names]
            raw.set_annotations(
                mne.Annotations(
                    onset=onset,
                    duration=duration,
                    description=description,
                    orig_time=raw.annotations.orig_time,
                    ch_names=ch_names,
                )
            )

        self.review["label_audit"] = label_audit
        self.review["channel_audit"] = [
            {
                "original_channel": old,
                "standard_channel": rename_map.get(old, old),
                "globally_bad": old in bad_old_names,
                "reason": reasons.get(old, ""),
            }
            for old in old_names
        ]
        return float(self.zoom_start.value()), float(self.zoom_end.value())


def open_metadata_review_gui(review):
    _qt_app()
    dialog = ReviewMetadataDialog(review)
    if dialog.exec() != QDialog.DialogCode.Accepted:
        raise RuntimeError("Metadata review was cancelled; nothing was changed.")
    return dialog.apply_changes()


def sync_legacy_eeg_dictionary(review, eeg):
    """Keep the notebook's later Plotly/spectrogram cells compatible."""
    raw = review["master"]
    eeg["labels"] = list(raw.ch_names)
    eeg["markers"] = [
        {
            "label": str(description),
            "sample": int(round(float(onset) * raw.info["sfreq"])) + 1,
            "time_sec": float(onset),
            "duration_sec": float(duration),
            "channels": list(channels),
        }
        for onset, duration, description, channels in zip(
            raw.annotations.onset,
            raw.annotations.duration,
            raw.annotations.description,
            raw.annotations.ch_names,
        )
    ]
    return eeg


class SaveReviewDialog(QDialog):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Save reviewed EEG")
        self.resize(520, 330)

        explanation = QLabel(
            "Use deidentified identifiers only. The source BrainVision files "
            "will not be changed. A new versioned folder will be created."
        )
        explanation.setWordWrap(True)

        self.participant = QLineEdit("L000")
        self.run = QLineEdit("run-01")
        self.reviewer = QLineEdit()
        self.context = QComboBox()
        self.context.addItems(
            ["preop1", "preop2", "or1", "or2", "or3", "or4"]
        )

        form = QFormLayout()
        form.addRow("Participant:", self.participant)
        form.addRow("Recording:", self.run)
        form.addRow("Reviewer initials/ID:", self.reviewer)
        form.addRow("File context:", self.context)

        buttons = QDialogButtonBox(
            QDialogButtonBox.StandardButton.Save
            | QDialogButtonBox.StandardButton.Cancel
        )
        buttons.button(QDialogButtonBox.StandardButton.Save).clicked.connect(
            self._validate_and_accept
        )
        buttons.rejected.connect(self.reject)

        layout = QVBoxLayout(self)
        layout.addWidget(explanation)
        layout.addLayout(form)
        layout.addWidget(buttons)

    def _validate_and_accept(self):
        participant = self.participant.text().strip()
        if not re.fullmatch(r"L[0-9]{3,}", participant, flags=re.IGNORECASE):
            QMessageBox.warning(
                self,
                "Invalid participant",
                "Use a deidentified participant ID such as L000 or L0123.",
            )
            return
        run = self.run.text().strip()
        if not re.fullmatch(r"run-[A-Za-z0-9]+", run):
            QMessageBox.warning(
                self,
                "Invalid recording",
                "Use a recording value such as run-01.",
            )
            return
        if not re.fullmatch(r"[A-Za-z0-9_-]{1,20}", self.reviewer.text().strip()):
            QMessageBox.warning(
                self,
                "Invalid reviewer",
                "Enter 1–20 initials/ID characters without spaces.",
            )
            return
        self.accept()

    def values(self):
        return {
            "participant": self.participant.text().strip(),
            "run": self.run.text().strip(),
            "reviewer": self.reviewer.text().strip(),
            "context": self.context.currentText(),
        }


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _write_tsv(path, rows, columns):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, delimiter="\t")
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row.get(column, "") for column in columns})


def save_review_gui(review, export_folder):
    """Collect deidentified metadata and save a versioned review bundle."""
    _qt_app()
    dialog = SaveReviewDialog()
    if dialog.exec() != QDialog.DialogCode.Accepted:
        raise RuntimeError("Save was cancelled.")
    values = dialog.values()

    participant = values["participant"]
    run = values["run"]
    reviewer = values["reviewer"]
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    context = values["context"]
    base = f"{participant}_task-anesthesia_acq-{context}_{run}"
    output = (
        Path(export_folder)
        / participant
        / "eeg"
        / f"review-{stamp}"
    )
    output.mkdir(parents=True, exist_ok=False)

    master = review["master"]
    processed = review["filtered"].copy().load_data()
    processed_data = processed.get_data().astype(float)
    channel_mean = np.nanmean(processed_data, axis=1, keepdims=True)
    channel_std = np.nanstd(processed_data, axis=1, keepdims=True)
    channel_std[~np.isfinite(channel_std) | (channel_std == 0)] = 1.0
    processed._data[:] = (processed_data - channel_mean) / channel_std
    processed.set_annotations(master.annotations.copy())
    processed.info["bads"] = list(master.info["bads"])

    reviewed_path = output / f"{base}_desc-filteredZscore_raw.fif"
    processed.save(reviewed_path, overwrite=False, verbose="ERROR")

    annotation_rows = []
    for onset, duration, description, channels in zip(
        master.annotations.onset,
        master.annotations.duration,
        master.annotations.description,
        master.annotations.ch_names,
    ):
        annotation_rows.append(
            {
                "onset_sec": float(onset),
                "duration_sec": float(duration),
                "standard_label": str(description),
                "channels": ",".join(channels) if channels else "all",
                "reviewer": reviewer,
                "review_version": stamp,
            }
        )
    _write_tsv(
        output / f"{base}_annotations.tsv",
        annotation_rows,
        [
            "onset_sec",
            "duration_sec",
            "standard_label",
            "channels",
            "reviewer",
            "review_version",
        ],
    )
    _write_tsv(
        output / f"{base}_label_audit.tsv",
        review["label_audit"],
        [
            "onset_sec",
            "duration_sec",
            "raw_label",
            "standard_label",
            "reviewer_note",
        ],
    )
    _write_tsv(
        output / f"{base}_channels.tsv",
        review["channel_audit"],
        [
            "original_channel",
            "standard_channel",
            "globally_bad",
            "reason",
        ],
    )

    summary = {
        "participant_id": participant,
        "run_id": run,
        "reviewer": reviewer,
        "file_context": values["context"],
        "review_created_utc": stamp,
        "source_extension": review["source"].suffix.lower(),
        "source_sha256": _sha256(review["source"]),
        "source_filename_omitted_to_reduce_phi_risk": True,
        "sampling_frequency_hz": float(master.info["sfreq"]),
        "n_channels": len(master.ch_names),
        "globally_bad_channels": list(master.info["bads"]),
        "n_annotations": len(master.annotations),
        "saved_signal": {
            "bandpass_hz": [0.5, 50.0],
            "normalization": "per-channel ordinary z-score",
            "saved_as_authoritative_signal": True,
            "data_unit": "z-score (dimensionless)",
            "power_spectral_density_unit": "z-score squared per Hz",
        },
        "ica_applied": False,
        "physical_unit_eeg_saved": False,
        "zscore_saved": True,
    }
    with (output / f"{base}_review.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    QMessageBox.information(
        None,
        "Review saved",
        "The reviewed 0.5–50 Hz filtered, per-channel z-scored EEG "
        "and audit tables were saved to:\n\n"
        f"{output}\n\n"
        "Reopen the reviewed FIF for verification before sharing.",
    )
    return output


def make_plotly_display_eeg(review, eeg):
    """Return the legacy EEG dictionary using the filtered physical signal."""
    eeg = sync_legacy_eeg_dictionary(review, eeg)
    display_eeg = dict(eeg)
    display_eeg["data"] = review["filtered"].get_data()
    display_eeg["labels"] = list(review["filtered"].ch_names)
    display_eeg["srate"] = float(review["filtered"].info["sfreq"])
    display_eeg["n_samples"] = int(review["filtered"].n_times)
    display_eeg["duration_sec"] = (
        float(review["filtered"].times[-1])
        if review["filtered"].n_times
        else 0.0
    )
    display_eeg["markers"] = list(eeg["markers"])
    return display_eeg


In [4]:
# Step 3: choose and load one queued recording.
# A Windows dialog asks which selected file to review.
# This creates:
#   review['master']   = original physical-unit EEG with editable metadata
#   review['filtered'] = separate 0.5–50 Hz physical-unit copy
#   review['display']  = filtered, per-channel z-scored GUI copy

review = prepare_gui_recording(VHDR_FILES)
VHDR_FILE = str(review["source"])

# Keep the original notebook dictionary for the later spectrogram/export cells.
eeg = load_brainvision(VHDR_FILE)

print("Loaded:", Path(VHDR_FILE).name)
print("Channels:", len(review["master"].ch_names), review["master"].ch_names)
print("Sampling rate:", review["master"].info["sfreq"], "Hz")
print("Duration:", round(review["master"].times[-1], 3), "seconds")
print("Existing annotations:", len(review["master"].annotations))
print("Ready for Step 4.")


Loaded: 20260626preop.vhdr
Channels: 8 ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
Sampling rate: 1000.0 Hz
Duration: 391.919 seconds
Existing annotations: 237
Ready for Step 4.


In [10]:
# Step 4: interactive Plotly/Dash EEG annotation interface.
#
# Processing order:
#   original EEG → 0.5–50 Hz filtering → per-channel z-score
#
# This interface allows:
#   • box-selection of artifact intervals;
#   • clicking to select point events;
#   • automatic conversion from plot minutes to exact seconds;
#   • standardized annotation labels;
#   • ALL-channel or channel-specific annotations;
#   • undoing newly added annotations.
#
# Z-scoring and clipping affect visualization only.

import importlib.util
import socket
import subprocess
import sys
import threading
import webbrowser

import numpy as np
import plotly.graph_objects as go

# Install Dash if it is not available.
if importlib.util.find_spec("dash") is None:

    print("Installing Dash...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "dash",
        ]
    )

from dash import (
    Dash,
    Input,
    Output,
    State,
    ctx,
    dash_table,
    dcc,
    html,
    no_update,
)

# ---------------------------------------------------------
# Prepare the filtered EEG.
# ---------------------------------------------------------

eeg_display = make_plotly_display_eeg(
    review,
    eeg,
)

data_filtered = eeg_display[
    "data"
].astype(float)

channel_labels = list(
    eeg_display["labels"]
)

sampling_rate = float(
    eeg_display["srate"]
)

number_samples = int(
    eeg_display["n_samples"]
)

print(
    "Display filtering:",
    review["filtered"].info["highpass"],
    "to",
    review["filtered"].info["lowpass"],
    "Hz",
)

# ---------------------------------------------------------
# Z-score each channel over the complete filtered recording.
# ---------------------------------------------------------

channel_mean = np.nanmean(
    data_filtered,
    axis=1,
    keepdims=True,
)

channel_std = np.nanstd(
    data_filtered,
    axis=1,
    keepdims=True,
)

channel_std[
    ~np.isfinite(channel_std)
    | (channel_std == 0)
] = 1.0

data_zscore = (
    data_filtered - channel_mean
) / channel_std

# ---------------------------------------------------------
# Downsample for browser visualization only.
# ---------------------------------------------------------

MAX_DISPLAY_POINTS = 20000

display_step = max(
    1,
    int(
        np.ceil(
            number_samples
            / MAX_DISPLAY_POINTS
        )
    ),
)

display_samples = np.arange(
    0,
    number_samples,
    display_step,
)

time_minutes = (
    display_samples
    / sampling_rate
    / 60.0
)

data_to_plot = data_zscore[
    :,
    display_samples,
]

# Display values only between −2 and +2 z.
DISPLAY_Z_LIMIT = 2.0

data_to_plot = np.clip(
    data_to_plot,
    -DISPLAY_Z_LIMIT,
    DISPLAY_Z_LIMIT,
)

CHANNEL_SPACING = 2.0

number_channels = len(
    channel_labels
)

offsets = (
    np.arange(number_channels)[::-1]
    * CHANNEL_SPACING
)

# ---------------------------------------------------------
# High-contrast channel colors.
# ---------------------------------------------------------

CHANNEL_COLORS = {
    "FP1": "#0072B2",
    "FP2": "#D55E00",
    "F7": "#009E73",
    "F8": "#CC79A7",
    "T7": "#E69F00",
    "T8": "#56B4E9",
    "O1": "#222222",
    "O2": "#7B2CBF",
}

FALLBACK_COLORS = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#CC79A7",
    "#E69F00",
    "#56B4E9",
    "#222222",
    "#7B2CBF",
]

top_of_plot = (
    offsets[0]
    + DISPLAY_Z_LIMIT
)

bottom_of_plot = (
    offsets[-1]
    - DISPLAY_Z_LIMIT
)

MARKER_LABEL_Y = (
    top_of_plot + 0.7
)

# Newly created Step 4 annotations are held here
# until Step 5 transfers them into the reviewed EEG.
STEP4_MANUAL_ANNOTATIONS = []

# Number already transferred to MNE.
STEP4_SYNCED_COUNT = 0


def build_interactive_eeg_figure():
    """Build the EEG figure with existing and newly added annotations."""

    figure = go.Figure()

    # Add EEG channels.
    for channel_index, channel_label in enumerate(
        channel_labels
    ):

        normalized_label = (
            channel_label
            .strip()
            .upper()
            .replace("EEG ", "")
        )

        channel_color = CHANNEL_COLORS.get(
            normalized_label,
            FALLBACK_COLORS[
                channel_index
                % len(FALLBACK_COLORS)
            ],
        )

        figure.add_trace(
            go.Scattergl(
                x=time_minutes,
                y=(
                    data_to_plot[
                        channel_index
                    ]
                    + offsets[
                        channel_index
                    ]
                ),
                mode="lines",
                name=channel_label,
                line={
                    "width": 1.5,
                    "color": channel_color,
                },
                customdata=data_to_plot[
                    channel_index
                ],
                hovertemplate=(
                    f"<b>{channel_label}</b><br>"
                    "Time: %{x:.4f} min<br>"
                    "Z-score: %{customdata:.2f}"
                    "<extra></extra>"
                ),
            )
        )

    # Combine imported annotations and new annotations.
    all_markers = []

    for marker in eeg_display["markers"]:

        all_markers.append(
            {
                "time_sec": float(
                    marker["time_sec"]
                ),
                "duration_sec": float(
                    marker.get(
                        "duration_sec",
                        0.0,
                    )
                ),
                "label": str(
                    marker["label"]
                ),
                "channels": marker.get(
                    "channels",
                    ["ALL"],
                ),
            }
        )

    all_markers.extend(
        STEP4_MANUAL_ANNOTATIONS
    )

    # Add annotation markers.
    for marker in all_markers:

        marker_time_sec = float(
            marker["time_sec"]
        )

        marker_duration_sec = float(
            marker.get(
                "duration_sec",
                0.0,
            )
        )

        marker_time_min = (
            marker_time_sec / 60.0
        )

        marker_label = str(
            marker["label"]
        )

        if marker_duration_sec > 0:

            marker_end_min = (
                marker_time_sec
                + marker_duration_sec
            ) / 60.0

            if marker_label.startswith(
                "BAD_"
            ):

                fill_color = (
                    "rgba(255, 0, 0, 0.15)"
                )

            elif marker_label.startswith(
                "CLEAN_"
            ):

                fill_color = (
                    "rgba(0, 170, 0, 0.10)"
                )

            else:

                fill_color = (
                    "rgba(30, 100, 255, 0.10)"
                )

            figure.add_vrect(
                x0=marker_time_min,
                x1=marker_end_min,
                fillcolor=fill_color,
                line_width=0,
                layer="below",
            )

        figure.add_vline(
            x=marker_time_min,
            line_color="red",
            line_dash="dash",
            line_width=1,
            opacity=0.65,
        )

        figure.add_annotation(
            x=marker_time_min,
            y=MARKER_LABEL_Y,
            text=marker_label,
            showarrow=False,
            textangle=-45,
            font={
                "color": "red",
                "size": 11,
            },
            yanchor="top",
        )

    figure.update_layout(
        title=(
            "EEG review: drag a box for an interval "
            "or click for a point event"
        ),
        xaxis_title=(
            "Time from beginning of recording "
            "(minutes)"
        ),
        yaxis={
            "title": "Channels",
            "tickmode": "array",
            "tickvals": offsets,
            "ticktext": channel_labels,
            "range": [
                bottom_of_plot - 1,
                top_of_plot + 1.5,
            ],
        },
        height=max(
            800,
            110 * number_channels,
        ),
        hovermode="closest",
        showlegend=False,
        dragmode="select",
        clickmode="event+select",
        uirevision="keep-eeg-view",
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin={
            "l": 110,
            "r": 30,
            "t": 85,
            "b": 60,
        },
    )

    figure.update_xaxes(
        showgrid=True,
        gridcolor=(
            "rgba(180, 180, 180, 0.25)"
        ),
    )

    figure.update_yaxes(
        showgrid=False,
    )

    return figure


# ---------------------------------------------------------
# Create the Dash application.
# ---------------------------------------------------------

step4_app = Dash(
    __name__,
)

channel_options = [
    {
        "label": "ALL channels",
        "value": "ALL",
    }
]

channel_options.extend(
    {
        "label": channel,
        "value": channel,
    }
    for channel in channel_labels
)

label_options = [
    {
        "label": label,
        "value": label,
    }
    for label in STANDARD_LABELS
]

step4_app.layout = html.Div(
    [
        html.H2(
            "Anesthesia EEG Manual Annotation"
        ),

        html.P(
            [
                "For an interval: choose ",
                html.B("Interval"),
                ", then drag a rectangular box horizontally over the EEG. ",
                "For a point event: choose ",
                html.B("Point"),
                ", then click the desired time."
            ]
        ),

        html.Div(
            [
                # Left control panel.
                html.Div(
                    [
                        html.Label(
                            "Annotation type"
                        ),

                        dcc.RadioItems(
                            id="annotation-kind",
                            options=[
                                {
                                    "label": "Interval",
                                    "value": "interval",
                                },
                                {
                                    "label": "Point",
                                    "value": "point",
                                },
                            ],
                            value="interval",
                            inline=True,
                        ),

                        html.Br(),

                        html.Label(
                            "Standard label"
                        ),

                        dcc.Dropdown(
                            id="standard-label",
                            options=label_options,
                            value="BAD_movement",
                            clearable=False,
                        ),

                        html.Br(),

                        html.Label(
                            "Affected channels"
                        ),

                        dcc.Dropdown(
                            id="affected-channels",
                            options=channel_options,
                            value=["ALL"],
                            multi=True,
                        ),

                        html.Br(),

                        html.Label(
                            "Start/onset in minutes"
                        ),

                        dcc.Input(
                            id="start-minutes",
                            type="number",
                            step=0.0001,
                            style={
                                "width": "100%",
                            },
                        ),

                        html.Br(),
                        html.Br(),

                        html.Label(
                            "End in minutes"
                        ),

                        dcc.Input(
                            id="end-minutes",
                            type="number",
                            step=0.0001,
                            style={
                                "width": "100%",
                            },
                        ),

                        html.Br(),
                        html.Br(),

                        html.Button(
                            "Add annotation",
                            id="add-annotation",
                            n_clicks=0,
                            style={
                                "fontWeight": "bold",
                                "marginRight": "8px",
                            },
                        ),

                        html.Button(
                            "Undo last",
                            id="undo-annotation",
                            n_clicks=0,
                        ),

                        html.Hr(),

                        html.Div(
                            id="selection-status",
                            children=(
                                "No time selected."
                            ),
                        ),

                        html.Div(
                            id="action-status",
                            style={
                                "marginTop": "12px",
                                "fontWeight": "bold",
                            },
                        ),
                    ],
                    style={
                        "width": "300px",
                        "minWidth": "300px",
                        "padding": "15px",
                        "border": (
                            "1px solid #cccccc"
                        ),
                        "borderRadius": "6px",
                        "backgroundColor": "#fafafa",
                    },
                ),

                # EEG graph.
                html.Div(
                    [
                        dcc.Graph(
                            id="eeg-graph",
                            figure=(
                                build_interactive_eeg_figure()
                            ),
                            config={
                                "displaylogo": False,
                                "scrollZoom": True,
                                "modeBarButtonsToRemove": [
                                    "lasso2d",
                                ],
                            },
                            style={
                                "height": (
                                    f"{max(800, 110 * number_channels)}px"
                                ),
                            },
                        )
                    ],
                    style={
                        "flex": "1",
                        "minWidth": "0",
                    },
                ),
            ],
            style={
                "display": "flex",
                "gap": "12px",
                "alignItems": "flex-start",
            },
        ),

        html.H3(
            "New annotations added in Step 4"
        ),

        dash_table.DataTable(
            id="manual-annotation-table",
            columns=[
                {
                    "name": "Onset (sec)",
                    "id": "time_sec",
                },
                {
                    "name": "Duration (sec)",
                    "id": "duration_sec",
                },
                {
                    "name": "Label",
                    "id": "label",
                },
                {
                    "name": "Channels",
                    "id": "channels_text",
                },
            ],
            data=[],
            style_table={
                "overflowX": "auto",
            },
            style_cell={
                "padding": "6px",
                "textAlign": "left",
            },
        ),

        html.P(
            [
                html.B("When finished: "),
                "return to VS Code and run Step 5."
            ]
        ),
    ],
    style={
        "fontFamily": "Arial, sans-serif",
        "padding": "15px",
    },
)


# ---------------------------------------------------------
# Capture interval or point selections.
# ---------------------------------------------------------

@step4_app.callback(
    Output(
        "start-minutes",
        "value",
    ),
    Output(
        "end-minutes",
        "value",
    ),
    Output(
        "selection-status",
        "children",
    ),
    Input(
        "eeg-graph",
        "selectedData",
    ),
    Input(
        "eeg-graph",
        "clickData",
    ),
    Input(
        "annotation-kind",
        "value",
    ),
    prevent_initial_call=True,
)
def capture_plot_selection(
    selected_data,
    click_data,
    annotation_kind,
):

    if annotation_kind == "interval":

        if not selected_data:

            return (
                no_update,
                no_update,
                (
                    "Drag a box over the "
                    "desired interval."
                ),
            )

        # Plotly normally provides the exact
        # box-selection x-axis range.
        selected_range = selected_data.get(
            "range",
            {},
        ).get(
            "x",
        )

        if selected_range:

            start_minute = float(
                min(selected_range)
            )

            end_minute = float(
                max(selected_range)
            )

        else:

            points = selected_data.get(
                "points",
                [],
            )

            if not points:

                return (
                    no_update,
                    no_update,
                    "No EEG points selected.",
                )

            selected_times = [
                float(point["x"])
                for point in points
            ]

            start_minute = min(
                selected_times
            )

            end_minute = max(
                selected_times
            )

        start_minute = round(
            start_minute,
            4,
        )

        end_minute = round(
            end_minute,
            4,
        )

        duration_minute = round(
            end_minute - start_minute,
            4,
        )

        return (
            start_minute,
            end_minute,
            (
                f"Selected interval: "
                f"{start_minute:.4f}–"
                f"{end_minute:.4f} minutes "
                f"(duration "
                f"{duration_minute:.4f} min)"
            ),
        )

    # Point-event mode.
    if not click_data:

        return (
            no_update,
            no_update,
            "Click the desired event time.",
        )

    clicked_minute = float(
        click_data["points"][0]["x"]
    )

    onset_minute = round(
        clicked_minute,
        4,
    )

    return (
        onset_minute,
        onset_minute,
        (
            f"Selected point: "
            f"{onset_minute:.4f} minutes"
        ),
    )


# ---------------------------------------------------------
# Add and undo annotations.
# ---------------------------------------------------------

@step4_app.callback(
    Output(
        "eeg-graph",
        "figure",
    ),
    Output(
        "manual-annotation-table",
        "data",
    ),
    Output(
        "action-status",
        "children",
    ),
    Input(
        "add-annotation",
        "n_clicks",
    ),
    Input(
        "undo-annotation",
        "n_clicks",
    ),
    State(
        "annotation-kind",
        "value",
    ),
    State(
        "standard-label",
        "value",
    ),
    State(
        "affected-channels",
        "value",
    ),
    State(
        "start-minutes",
        "value",
    ),
    State(
        "end-minutes",
        "value",
    ),
    prevent_initial_call=True,
)
def modify_manual_annotations(
    add_clicks,
    undo_clicks,
    annotation_kind,
    standard_label,
    affected_channels,
    start_minute,
    end_minute,
):

    triggered = ctx.triggered_id

    if triggered == "undo-annotation":

        if STEP4_MANUAL_ANNOTATIONS:

            removed = (
                STEP4_MANUAL_ANNOTATIONS.pop()
            )

            status = (
                "Removed last annotation: "
                f"{removed['label']}"
            )

        else:

            status = (
                "There are no new "
                "annotations to undo."
            )

    elif triggered == "add-annotation":

        if start_minute is None:

            return (
                no_update,
                no_update,
                "Select a time first.",
            )

        if not affected_channels:

            return (
                no_update,
                no_update,
                "Select ALL or affected channels.",
            )

        start_minute = float(
            start_minute
        )

        if annotation_kind == "interval":

            if end_minute is None:

                return (
                    no_update,
                    no_update,
                    "Select an interval end time.",
                )

            end_minute = float(
                end_minute
            )

            if end_minute <= start_minute:

                return (
                    no_update,
                    no_update,
                    (
                        "Interval end must be "
                        "after interval start."
                    ),
                )

            duration_minute = (
                end_minute - start_minute
            )

        else:

            duration_minute = 0.0

        # MNE and the saved HDF5 use seconds internally.
        # Only the Step 4 entry boxes use minutes.
        start_sec = start_minute * 60.0
        duration_sec = duration_minute * 60.0

        if "ALL" in affected_channels:

            saved_channels = ["ALL"]

        else:

            saved_channels = list(
                affected_channels
            )

        annotation = {
            "time_sec": round(
                start_sec,
                3,
            ),
            "duration_sec": round(
                duration_sec,
                3,
            ),
            "label": str(
                standard_label
            ),
            "channels": saved_channels,
        }

        STEP4_MANUAL_ANNOTATIONS.append(
            annotation
        )

        status = (
            f"Added {standard_label} "
            f"at {start_minute:.4f} minutes."
        )

    else:

        return (
            no_update,
            no_update,
            no_update,
        )

    table_rows = []

    for annotation in (
        STEP4_MANUAL_ANNOTATIONS
    ):

        table_rows.append(
            {
                "time_sec": (
                    annotation["time_sec"]
                ),
                "duration_sec": (
                    annotation[
                        "duration_sec"
                    ]
                ),
                "label": annotation["label"],
                "channels_text": ",".join(
                    annotation["channels"]
                ),
            }
        )

    return (
        build_interactive_eeg_figure(),
        table_rows,
        status,
    )


# ---------------------------------------------------------
# Find an available local port and launch the browser app.
# ---------------------------------------------------------

with socket.socket() as port_socket:

    port_socket.bind(
        ("127.0.0.1", 0)
    )

    STEP4_DASH_PORT = int(
        port_socket.getsockname()[1]
    )

STEP4_DASH_URL = (
    f"http://127.0.0.1:"
    f"{STEP4_DASH_PORT}"
)

print(
    "Starting interactive EEG annotation interface:"
)

print(
    STEP4_DASH_URL
)

# Open the browser shortly after the server begins.
threading.Timer(
    1.2,
    lambda: webbrowser.open(
        STEP4_DASH_URL
    ),
).start()

step4_app.run(
    host="127.0.0.1",
    port=STEP4_DASH_PORT,
    debug=False,
    jupyter_mode="external",
)

print(
    "After completing browser annotations, "
    "run Step 5."
)

Display filtering: 0.5 to 50.0 Hz
Starting interactive EEG annotation interface:
http://127.0.0.1:59574
Dash app running on http://127.0.0.1:59574/
After completing browser annotations, run Step 5.


In [9]:
# Step 5: batch-review annotations, reopen the full EEG, and save one HDF5 file.
#
# Requirements:
#   • Run Steps 1–4 first.
#   • Step 4 must have created STEP4_MANUAL_ANNOTATIONS.
#   • h5py must be installed in the notebook environment.
#
# The single saved .h5 file contains:
#   • 0.5–50 Hz filtered, independently per-channel z-scored EEG;
#   • sampling rate and channel labels;
#   • globally bad-channel labels;
#   • annotations, including channel-specific annotations;
#   • label, channel, and deletion audit metadata;
#   • participant, context, reviewer, and processing metadata.
#
# Display clipping is NOT applied to the saved EEG.

import json
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from PyQt6.QtCore import QItemSelectionModel
from PyQt6.QtWidgets import (
    QAbstractItemView,
    QComboBox,
    QDialog,
    QDialogButtonBox,
    QFormLayout,
    QGroupBox,
    QHBoxLayout,
    QLabel,
    QLineEdit,
    QMessageBox,
    QPushButton,
    QVBoxLayout,
)

try:
    import h5py
except ImportError as error:
    raise ImportError(
        "Step 5 requires h5py. Run `%pip install h5py` in a notebook cell, "
        "restart the kernel, and rerun Steps 1–5."
    ) from error


pio.renderers.default = "browser"


# ---------------------------------------------------------
# Transfer new Step 4 annotations without creating duplicates.
# ---------------------------------------------------------

def annotation_already_exists(
    raw_copy,
    onset_sec,
    duration_sec,
    standard_label,
    annotation_channels,
):
    """Return True if an identical annotation is already present."""

    expected_channels = set(annotation_channels)

    for (
        existing_onset,
        existing_duration,
        existing_label,
        existing_channels,
    ) in zip(
        raw_copy.annotations.onset,
        raw_copy.annotations.duration,
        raw_copy.annotations.description,
        raw_copy.annotations.ch_names,
    ):
        if (
            np.isclose(float(existing_onset), float(onset_sec), atol=0.001)
            and np.isclose(
                float(existing_duration),
                float(duration_sec),
                atol=0.001,
            )
            and str(existing_label) == str(standard_label)
            and set(existing_channels) == expected_channels
        ):
            return True

    return False


if "STEP4_MANUAL_ANNOTATIONS" not in globals():
    raise RuntimeError("Run Step 4 before running Step 5.")

if "STEP4_SYNCED_COUNT" not in globals():
    STEP4_SYNCED_COUNT = 0

new_annotations = STEP4_MANUAL_ANNOTATIONS[STEP4_SYNCED_COUNT:]
transferred_count = 0
skipped_duplicate_count = 0

for annotation in new_annotations:
    onset_sec = float(annotation["time_sec"])
    duration_sec = float(annotation["duration_sec"])
    standard_label = str(annotation["label"])
    selected_channels = list(annotation["channels"])

    # An empty MNE channel list means that the annotation affects all channels.
    annotation_channels = (
        []
        if "ALL" in selected_channels
        else list(selected_channels)
    )

    for raw_copy in (
        review["master"],
        review["filtered"],
        review["display"],
    ):
        if annotation_already_exists(
            raw_copy=raw_copy,
            onset_sec=onset_sec,
            duration_sec=duration_sec,
            standard_label=standard_label,
            annotation_channels=annotation_channels,
        ):
            skipped_duplicate_count += 1
            continue

        raw_copy.annotations.append(
            onset=[onset_sec],
            duration=[duration_sec],
            description=[standard_label],
            ch_names=[annotation_channels],
        )
        transferred_count += 1

STEP4_SYNCED_COUNT = len(STEP4_MANUAL_ANNOTATIONS)

print(f"Processed {len(new_annotations)} new Step 4 annotations.")
print(
    f"Added {transferred_count} annotation copies across "
    "master, filtered, and display data."
)
if skipped_duplicate_count:
    print(f"Skipped {skipped_duplicate_count} duplicate copies.")


# ---------------------------------------------------------
# Configure the existing metadata dialog for Step 5.
#
# The insertion and optional zoom sections are hidden.
# Rows use extended row selection:
#   click the first row, hold Shift, click the last row.
# ---------------------------------------------------------

_qt_app()
metadata_dialog = ReviewMetadataDialog(review)
metadata_dialog.setWindowTitle("Step 5: final EEG metadata review")

for group_box in metadata_dialog.findChildren(QGroupBox):
    if group_box.title() in {
        "Insert a new annotation",
        "Optional range to reopen in the EEG viewer",
    }:
        group_box.hide()

instruction_labels = metadata_dialog.findChildren(QLabel)
if instruction_labels:
    instruction_labels[0].setText(
        "Review channel names, globally bad channels, and existing annotations. "
        "To delete a continuous block of annotation rows, click the first row, "
        "hold Shift, click the last row, then click Delete selected rows. "
        "Use Select all Primary/P 1 for the repeated device-error labels."
    )

annotation_table = metadata_dialog.annotation_table
annotation_table.setSelectionBehavior(
    QAbstractItemView.SelectionBehavior.SelectRows
)
annotation_table.setSelectionMode(
    QAbstractItemView.SelectionMode.ExtendedSelection
)

# The old individual Delete checkboxes are hidden because deletion is now
# performed with row selection and the batch-delete button.
annotation_table.setColumnHidden(5, True)

deleted_annotation_audit = []


def normalized_annotation_label(value):
    """Normalize spacing and case for matching repeated device labels."""

    return re.sub(r"\s+", "", str(value)).casefold()


def select_rows(rows):
    """Select complete table rows without clearing earlier selections."""

    selection_model = annotation_table.selectionModel()
    flags = (
        QItemSelectionModel.SelectionFlag.Select
        | QItemSelectionModel.SelectionFlag.Rows
    )

    for row in rows:
        selection_model.select(annotation_table.model().index(row, 0), flags)


def select_all_primary_p1():
    """Select every Primary/P 1 row, even if whitespace varies."""

    target = normalized_annotation_label("Primary/P 1")
    matching_rows = []

    for row in range(annotation_table.rowCount()):
        item = annotation_table.item(row, 2)
        if item and normalized_annotation_label(item.text()) == target:
            matching_rows.append(row)

    annotation_table.clearSelection()
    select_rows(matching_rows)

    QMessageBox.information(
        metadata_dialog,
        "Primary/P 1 selection",
        f"Selected {len(matching_rows)} Primary/P 1 rows.\n\n"
        "Click Delete selected rows to remove them.",
    )


def select_all_matching_selected_label():
    """Select every row having the same original label as the active row."""

    selected_rows = annotation_table.selectionModel().selectedRows()
    if not selected_rows:
        QMessageBox.warning(
            metadata_dialog,
            "No row selected",
            "Select one annotation row first.",
        )
        return

    active_row = selected_rows[0].row()
    active_item = annotation_table.item(active_row, 2)
    if active_item is None:
        return

    target = normalized_annotation_label(active_item.text())
    matching_rows = []

    for row in range(annotation_table.rowCount()):
        item = annotation_table.item(row, 2)
        if item and normalized_annotation_label(item.text()) == target:
            matching_rows.append(row)

    annotation_table.clearSelection()
    select_rows(matching_rows)


def delete_selected_annotation_rows():
    """Remove selected annotation rows after confirmation."""

    rows = sorted(
        {
            index.row()
            for index in annotation_table.selectionModel().selectedRows()
        },
        reverse=True,
    )

    if not rows:
        QMessageBox.warning(
            metadata_dialog,
            "No rows selected",
            "Select one or more annotation rows first.",
        )
        return

    answer = QMessageBox.question(
        metadata_dialog,
        "Delete selected annotations",
        f"Delete {len(rows)} selected annotation rows?",
        QMessageBox.StandardButton.Yes | QMessageBox.StandardButton.No,
        QMessageBox.StandardButton.No,
    )

    if answer != QMessageBox.StandardButton.Yes:
        return

    for row in rows:
        deleted_annotation_audit.append(
            {
                "onset_sec": annotation_table.item(row, 0).text(),
                "duration_sec": annotation_table.item(row, 1).text(),
                "original_label": annotation_table.item(row, 2).text(),
                "channels": annotation_table.item(row, 4).text(),
                "action": "deleted_during_step5",
            }
        )
        annotation_table.removeRow(row)


def clear_annotation_selection():
    annotation_table.clearSelection()


batch_box = QGroupBox("Batch annotation selection and deletion")
batch_layout = QVBoxLayout(batch_box)

batch_instructions = QLabel(
    "For a continuous range: click the first row, hold Shift, and click "
    "the last row. You can also select every repeated device-error label."
)
batch_instructions.setWordWrap(True)
batch_layout.addWidget(batch_instructions)

batch_buttons = QHBoxLayout()

select_primary_button = QPushButton("Select all Primary/P 1")
select_primary_button.clicked.connect(select_all_primary_p1)
batch_buttons.addWidget(select_primary_button)

select_matching_button = QPushButton("Select all matching selected label")
select_matching_button.clicked.connect(
    select_all_matching_selected_label
)
batch_buttons.addWidget(select_matching_button)

delete_rows_button = QPushButton("Delete selected rows")
delete_rows_button.clicked.connect(delete_selected_annotation_rows)
batch_buttons.addWidget(delete_rows_button)

clear_selection_button = QPushButton("Clear selection")
clear_selection_button.clicked.connect(clear_annotation_selection)
batch_buttons.addWidget(clear_selection_button)

batch_layout.addLayout(batch_buttons)

# Insert the batch controls immediately above Apply/Cancel.
dialog_layout = metadata_dialog.layout()
dialog_layout.insertWidget(dialog_layout.count() - 1, batch_box)

if metadata_dialog.exec() != QDialog.DialogCode.Accepted:
    raise RuntimeError("Metadata review was cancelled; nothing was saved.")

# Apply the remaining rows, standardized labels, channel names, and bad flags.
metadata_dialog.apply_changes()
review["deleted_annotation_audit"] = deleted_annotation_audit

eeg = sync_legacy_eeg_dictionary(review, eeg)
eeg_display = make_plotly_display_eeg(review, eeg)

print()
print("Metadata changes applied.")
print("Standard channel labels:", review["master"].ch_names)
print("Globally bad channels:", review["master"].info["bads"] or "none")
print("Deleted annotation rows:", len(deleted_annotation_audit))


# ---------------------------------------------------------
# Reopen the complete updated EEG using a spacious display.
#
# Processing:
#   0.5–50 Hz filtered physical signal -> per-channel z-score.
#
# Only the display is clipped to ±2 z. The saved z-score data are not clipped.
# ---------------------------------------------------------

def build_full_review_figure(review_object, eeg_dictionary):
    """Build a full-range confirmation plot matching the Step 4 style."""

    display_eeg = make_plotly_display_eeg(
        review_object,
        eeg_dictionary,
    )

    filtered_data = display_eeg["data"].astype(float)
    labels = list(display_eeg["labels"])
    sampling_rate = float(display_eeg["srate"])
    number_samples = int(display_eeg["n_samples"])

    means = np.nanmean(filtered_data, axis=1, keepdims=True)
    standard_deviations = np.nanstd(
        filtered_data,
        axis=1,
        keepdims=True,
    )
    standard_deviations[
        ~np.isfinite(standard_deviations)
        | (standard_deviations == 0)
    ] = 1.0

    zscore_data = (
        filtered_data - means
    ) / standard_deviations

    maximum_display_points = 20000
    display_step = max(
        1,
        int(np.ceil(number_samples / maximum_display_points)),
    )
    display_samples = np.arange(
        0,
        number_samples,
        display_step,
    )
    time_minutes = (
        display_samples / sampling_rate / 60.0
    )

    display_z_limit = 2.0
    display_data = np.clip(
        zscore_data[:, display_samples],
        -display_z_limit,
        display_z_limit,
    )

    channel_spacing = 2.0
    offsets = (
        np.arange(len(labels))[::-1]
        * channel_spacing
    )

    channel_colors = {
        "FP1": "#0072B2",
        "FP2": "#D55E00",
        "F3": "#009E73",
        "F4": "#CC79A7",
        "F7": "#009E73",
        "F8": "#CC79A7",
        "T7": "#E69F00",
        "T8": "#56B4E9",
        "O1": "#222222",
        "O2": "#7B2CBF",
    }
    fallback_colors = [
        "#0072B2",
        "#D55E00",
        "#009E73",
        "#CC79A7",
        "#E69F00",
        "#56B4E9",
        "#222222",
        "#7B2CBF",
    ]

    figure = go.Figure()

    for channel_index, channel_label in enumerate(labels):
        normalized_label = (
            channel_label.strip().upper().replace("EEG ", "")
        )
        color = channel_colors.get(
            normalized_label,
            fallback_colors[channel_index % len(fallback_colors)],
        )

        figure.add_trace(
            go.Scattergl(
                x=time_minutes,
                y=display_data[channel_index] + offsets[channel_index],
                mode="lines",
                name=channel_label,
                line={"width": 1.5, "color": color},
                customdata=display_data[channel_index],
                hovertemplate=(
                    f"<b>{channel_label}</b><br>"
                    "Time: %{x:.4f} min<br>"
                    "Z-score: %{customdata:.2f}"
                    "<extra></extra>"
                ),
            )
        )

    top_of_plot = offsets[0] + display_z_limit
    bottom_of_plot = offsets[-1] - display_z_limit
    marker_label_y = top_of_plot + 0.7

    for marker in display_eeg["markers"]:
        marker_time_sec = float(marker["time_sec"])
        marker_duration_sec = float(
            marker.get("duration_sec", 0.0)
        )
        marker_time_min = marker_time_sec / 60.0
        marker_label = str(marker["label"])

        if marker_duration_sec > 0:
            marker_end_min = (
                marker_time_sec + marker_duration_sec
            ) / 60.0

            if marker_label.startswith("BAD_"):
                fill_color = "rgba(255, 0, 0, 0.15)"
            elif marker_label.startswith("CLEAN_"):
                fill_color = "rgba(0, 170, 0, 0.10)"
            else:
                fill_color = "rgba(30, 100, 255, 0.10)"

            figure.add_vrect(
                x0=marker_time_min,
                x1=marker_end_min,
                fillcolor=fill_color,
                line_width=0,
                layer="below",
            )

        figure.add_vline(
            x=marker_time_min,
            line_color="red",
            line_dash="dash",
            line_width=1,
            opacity=0.65,
        )
        figure.add_annotation(
            x=marker_time_min,
            y=marker_label_y,
            text=marker_label,
            showarrow=False,
            textangle=-45,
            font={"color": "red", "size": 11},
            yanchor="top",
        )

    figure.update_layout(
        title=(
            "Final EEG review: 0.5–50 Hz filtered, "
            "per-channel z-scored display"
        ),
        xaxis_title="Time from beginning of recording (minutes)",
        yaxis={
            "title": "Channels",
            "tickmode": "array",
            "tickvals": offsets,
            "ticktext": labels,
            "range": [
                bottom_of_plot - 1,
                top_of_plot + 1.5,
            ],
        },
        height=max(800, 110 * len(labels)),
        hovermode="closest",
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin={"l": 110, "r": 30, "t": 85, "b": 60},
    )
    figure.update_xaxes(
        showgrid=True,
        gridcolor="rgba(180, 180, 180, 0.25)",
    )
    figure.update_yaxes(showgrid=False)

    return figure


confirmation_figure = build_full_review_figure(review, eeg)
confirmation_figure.show()

print()
print("The complete updated EEG opened in the browser.")
print("Review the full recording before answering the confirmation box.")

answer = QMessageBox.question(
    None,
    "Confirm reviewed EEG",
    "Does the complete updated EEG and metadata look ready to save?",
    QMessageBox.StandardButton.Yes | QMessageBox.StandardButton.No,
    QMessageBox.StandardButton.No,
)

if answer != QMessageBox.StandardButton.Yes:
    raise RuntimeError(
        "Save cancelled. Rerun Step 5 to make corrections."
    )


# ---------------------------------------------------------
# Collect save metadata.
#
# There is no Surgical encounter field and no option to save
# a separate physical-unit EEG.
# ---------------------------------------------------------

class SaveProcessedEEGDialog(QDialog):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Save reviewed EEG")
        self.resize(520, 270)

        explanation = QLabel(
            "One HDF5 file will be saved. It contains the reviewed "
            "0.5–50 Hz filtered, per-channel z-scored EEG and all "
            "review metadata. Use deidentified values only."
        )
        explanation.setWordWrap(True)

        self.participant = QLineEdit("L000")
        self.reviewer = QLineEdit()
        self.context = QComboBox()
        self.context.addItems(
            ["preop1", "preop2", "or1", "or2", "or3", "or4"]
        )

        form = QFormLayout()
        form.addRow("Participant:", self.participant)
        form.addRow("Reviewer initials/ID:", self.reviewer)
        form.addRow("File context:", self.context)

        buttons = QDialogButtonBox(
            QDialogButtonBox.StandardButton.Save
            | QDialogButtonBox.StandardButton.Cancel
        )
        buttons.button(
            QDialogButtonBox.StandardButton.Save
        ).clicked.connect(self.validate_and_accept)
        buttons.rejected.connect(self.reject)

        layout = QVBoxLayout(self)
        layout.addWidget(explanation)
        layout.addLayout(form)
        layout.addWidget(buttons)

    def validate_and_accept(self):
        participant = self.participant.text().strip()
        reviewer = self.reviewer.text().strip()

        if not re.fullmatch(
            r"L[0-9]{3,}",
            participant,
            flags=re.IGNORECASE,
        ):
            QMessageBox.warning(
                self,
                "Invalid participant",
                "Use a deidentified participant ID such as L000 or L0123.",
            )
            return

        if not re.fullmatch(r"[A-Za-z0-9_-]{1,20}", reviewer):
            QMessageBox.warning(
                self,
                "Invalid reviewer",
                "Enter 1–20 initials/ID characters without spaces.",
            )
            return

        self.accept()

    def values(self):
        return {
            "participant": self.participant.text().strip().upper(),
            "reviewer": self.reviewer.text().strip(),
            "context": self.context.currentText(),
        }


def json_default(value):
    """Convert NumPy values to ordinary JSON-compatible values."""

    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


def save_single_review_hdf5(review_object, export_folder):
    """Save the processed EEG and all review metadata in one HDF5 file."""

    _qt_app()
    save_dialog = SaveProcessedEEGDialog()

    if save_dialog.exec() != QDialog.DialogCode.Accepted:
        raise RuntimeError("Save was cancelled.")

    values = save_dialog.values()
    participant = values["participant"]
    context = values["context"]
    reviewer = values["reviewer"]

    created_time = datetime.now(timezone.utc)
    created_iso = created_time.isoformat()

    export_path = Path(export_folder)
    export_path.mkdir(parents=True, exist_ok=True)

    output_file = export_path / (
        f"{participant}_{context}_reviewed.h5"
    )

    # A stable filename is convenient, but it must never silently replace
    # a reviewed recording that has not yet been packaged in Step 6.
    if output_file.exists():
        QMessageBox.warning(
            None,
            "Reviewed file already exists",
            (
                f"This reviewed file already exists:\n\n{output_file}\n\n"
                "Run Step 6 to package and remove it before saving "
                "another review with the same participant and context."
            ),
        )
        raise FileExistsError(output_file)

    filtered_raw = review_object["filtered"].copy().load_data()
    filtered_data = filtered_raw.get_data().astype(float)

    channel_mean = np.nanmean(
        filtered_data,
        axis=1,
        keepdims=True,
    )
    channel_std = np.nanstd(
        filtered_data,
        axis=1,
        keepdims=True,
    )
    channel_std[
        ~np.isfinite(channel_std)
        | (channel_std == 0)
    ] = 1.0

    # These are the saved values. They are not clipped to the display limit.
    processed_zscore = (
        (filtered_data - channel_mean)
        / channel_std
    ).astype(np.float32)

    master = review_object["master"]
    text_type = h5py.string_dtype(encoding="utf-8")

    with h5py.File(output_file, "w") as h5_file:
        h5_file.attrs["format_name"] = (
            "Anesthesia EEG reviewed single-file export"
        )
        h5_file.attrs["schema_version"] = "1.0"
        h5_file.attrs["participant_id"] = participant
        h5_file.attrs["file_context"] = context
        h5_file.attrs["reviewer"] = reviewer
        h5_file.attrs["review_created_utc"] = created_iso

        eeg_group = h5_file.create_group("eeg")
        chunk_samples = max(
            1,
            min(
                processed_zscore.shape[1],
                int(round(float(master.info["sfreq"]) * 10)),
            ),
        )
        eeg_dataset = eeg_group.create_dataset(
            "data_zscore",
            data=processed_zscore,
            dtype=np.float32,
            chunks=(1, chunk_samples),
            compression="gzip",
            compression_opts=4,
            shuffle=True,
        )
        eeg_dataset.attrs["dimensions"] = "channels x samples"
        eeg_dataset.attrs["unit"] = "z-score (dimensionless)"
        eeg_dataset.attrs["display_clipping_applied"] = False

        eeg_group.attrs["sampling_frequency_hz"] = float(
            master.info["sfreq"]
        )
        eeg_group.attrs["n_channels"] = int(
            processed_zscore.shape[0]
        )
        eeg_group.attrs["n_samples"] = int(
            processed_zscore.shape[1]
        )
        eeg_group.attrs["bandpass_low_hz"] = 0.5
        eeg_group.attrs["bandpass_high_hz"] = 50.0
        eeg_group.attrs["normalization"] = (
            "ordinary z-score independently per channel "
            "over the complete filtered recording"
        )
        eeg_group.attrs["power_spectral_density_unit"] = (
            "z-score squared per Hz"
        )

        eeg_group.create_dataset(
            "channel_labels",
            data=np.asarray(master.ch_names, dtype=object),
            dtype=text_type,
        )
        eeg_group.create_dataset(
            "globally_bad_channels",
            data=np.asarray(master.info["bads"], dtype=object),
            dtype=text_type,
        )

        annotation_group = h5_file.create_group("annotations")
        annotation_group.create_dataset(
            "onset_sec",
            data=np.asarray(master.annotations.onset, dtype=np.float64),
        )
        annotation_group.create_dataset(
            "duration_sec",
            data=np.asarray(master.annotations.duration, dtype=np.float64),
        )
        annotation_group.create_dataset(
            "label",
            data=np.asarray(
                [str(value) for value in master.annotations.description],
                dtype=object,
            ),
            dtype=text_type,
        )
        annotation_group.create_dataset(
            "channels",
            data=np.asarray(
                [
                    ",".join(channels) if channels else "all"
                    for channels in master.annotations.ch_names
                ],
                dtype=object,
            ),
            dtype=text_type,
        )

        review_group = h5_file.create_group("review")
        review_group.create_dataset(
            "label_audit_json",
            data=json.dumps(
                review_object.get("label_audit", []),
                default=json_default,
                indent=2,
            ),
            dtype=text_type,
        )
        review_group.create_dataset(
            "channel_audit_json",
            data=json.dumps(
                review_object.get("channel_audit", []),
                default=json_default,
                indent=2,
            ),
            dtype=text_type,
        )
        review_group.create_dataset(
            "deleted_annotations_json",
            data=json.dumps(
                review_object.get("deleted_annotation_audit", []),
                default=json_default,
                indent=2,
            ),
            dtype=text_type,
        )
        review_group.create_dataset(
            "processing_description",
            data=(
                "EEG was band-pass filtered from 0.5 to 50 Hz and "
                "then independently z-scored within each channel. "
                "No ICA was applied. Display clipping was not applied "
                "to the saved data. The physical-unit EEG is not "
                "included in this collaborator-facing file."
            ),
            dtype=text_type,
        )

    QMessageBox.information(
        None,
        "Review saved",
        "The processed EEG and all review metadata were saved in "
        f"one HDF5 file:\n\n{output_file}",
    )

    return output_file


SAVED_REVIEW_FILE = save_single_review_hdf5(
    review,
    EXPORT_FOLDER,
)

print()
print("Step 5 complete.")
print("Saved one HDF5 file:", SAVED_REVIEW_FILE)
print("Standard channel labels:", review["master"].ch_names)
print("Globally bad channels:", review["master"].info["bads"] or "none")
print("Total retained annotations:", len(review["master"].annotations))
print("Deleted annotation rows:", len(deleted_annotation_audit))
print(
    "Saved signal: 0.5–50 Hz filtered, per-channel z-scored, "
    "not display-clipped."
)
print("Saved PSD unit: z-score squared per Hz.")


Processed 0 new Step 4 annotations.
Added 0 annotation copies across master, filtered, and display data.

Metadata changes applied.
Standard channel labels: ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
Globally bad channels: ['C3', 'O1']
Deleted annotation rows: 0

The complete updated EEG opened in the browser.
Review the full recording before answering the confirmation box.


RuntimeError: Save cancelled. Rerun Step 5 to make corrections.

In [8]:
# Step 6: safely add the newly reviewed recording to one participant package.
#
# Run this cell after Step 5 finishes successfully.
#
# The participant package uses this structure:
#
#   L000_reviewed_eeg.h5
#   └── recordings
#       ├── preop1
#       ├── or1
#       └── or2
#
# Recordings remain independent; EEG signals are never concatenated.
# Existing recording groups are never overwritten.
# The participant package is updated through a validated temporary copy.
# After successful packaging, the separate Step 5 HDF5 file is removed.

import os
import re
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path

import h5py

from PyQt6.QtWidgets import QFileDialog, QMessageBox


def hdf5_attribute_text(value):
    """Return an HDF5 text attribute as an ordinary Python string."""

    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def safe_hdf5_group_name(value):
    """Create a readable HDF5 group name from reviewed metadata."""

    cleaned = re.sub(
        r"[^A-Za-z0-9_-]+",
        "-",
        str(value).strip(),
    ).strip("-_")

    if not cleaned:
        raise ValueError("The recording group name is empty.")

    return cleaned


def choose_reviewed_recording_file():
    """Use the Step 5 output, or ask the user to select a reviewed file."""

    step5_file = globals().get("SAVED_REVIEW_FILE")

    if step5_file is not None:
        step5_path = Path(step5_file)
        if step5_path.exists():
            return step5_path

    selected_file, _ = QFileDialog.getOpenFileName(
        None,
        "Select one individual reviewed recording from Step 5",
        str(EXPORT_FOLDER),
        "Individual reviewed EEG (*_reviewed.h5)",
    )

    if not selected_file:
        raise RuntimeError("No reviewed recording was selected.")

    return Path(selected_file)


def append_reviewed_recording_to_participant_package(
    reviewed_recording_file,
    export_folder,
):
    """Safely copy one reviewed recording into its participant HDF5."""

    source_file = Path(reviewed_recording_file).resolve()
    export_path = Path(export_folder).resolve()
    export_path.mkdir(parents=True, exist_ok=True)

    if not source_file.exists():
        raise FileNotFoundError(source_file)

    # Read and validate the individual Step 5 file.
    with h5py.File(source_file, "r") as source_h5:
        missing_groups = [
            name
            for name in ("eeg", "annotations", "review")
            if name not in source_h5
        ]

        if missing_groups:
            raise ValueError(
                "The selected file is not a complete Step 5 export. "
                f"Missing groups: {missing_groups}"
            )

        participant = hdf5_attribute_text(
            source_h5.attrs.get("participant_id", "")
        ).upper()
        context = hdf5_attribute_text(
            source_h5.attrs.get("file_context", "")
        ).lower()
    if not re.fullmatch(r"L[0-9]{3,}", participant):
        raise ValueError(
            "The reviewed file has an invalid participant ID: "
            f"{participant!r}"
        )

    allowed_contexts = {
        "preop1",
        "preop2",
        "or1",
        "or2",
        "or3",
        "or4",
    }

    if context not in allowed_contexts:
        raise ValueError(
            "The reviewed file has an invalid context: "
            f"{context!r}"
        )

    recording_group_name = safe_hdf5_group_name(context)

    # Recognize the older group naming convention so that a context
    # already stored as, for example, or1_run-01 cannot be added again.
    legacy_recording_group_name = safe_hdf5_group_name(
        f"{context}_run-01"
    )
    participant_file = export_path / f"{participant}_reviewed_eeg.h5"

    if source_file == participant_file.resolve():
        raise ValueError(
            "Select an individual Step 5 file, not the participant package."
        )

    # Validate an existing package before asking for confirmation.
    if participant_file.exists():
        with h5py.File(participant_file, "r") as package_h5:
            package_participant = hdf5_attribute_text(
                package_h5.attrs.get("participant_id", "")
            ).upper()

            if package_participant != participant:
                raise ValueError(
                    "Participant mismatch: the recording is "
                    f"{participant}, but the package is "
                    f"{package_participant}."
                )

            if "recordings" not in package_h5:
                raise ValueError(
                    "The existing destination is not a participant package."
                )

            existing_recording_names = set(
                package_h5["recordings"].keys()
            )

            duplicate_name = None
            if recording_group_name in existing_recording_names:
                duplicate_name = recording_group_name
            elif legacy_recording_group_name in existing_recording_names:
                duplicate_name = legacy_recording_group_name

            if duplicate_name is not None:
                QMessageBox.warning(
                    None,
                    "Recording already packaged",
                    (
                        f"{duplicate_name} already represents "
                        f"the {context} recording in:\n\n"
                        f"{participant_file}\n\n"
                        "Nothing was overwritten."
                    ),
                )
                raise RuntimeError(
                    f"Recording context already exists: {duplicate_name}"
                )

    answer = QMessageBox.question(
        None,
        "Add recording to participant package",
        (
            f"Participant ID read from reviewed file: {participant}\n"
            f"Recording group: {recording_group_name}\n\n"
            f"Source reviewed file:\n{source_file}\n\n"
            f"Exact matching participant package:\n"
            f"{participant_file}\n\n"
            "Packages belonging to any other participant in this folder "
            "will be ignored. The package's embedded participant ID must "
            "also match exactly.\n\n"
            "Add this recording?"
        ),
        QMessageBox.StandardButton.Yes
        | QMessageBox.StandardButton.No,
        QMessageBox.StandardButton.No,
    )

    if answer != QMessageBox.StandardButton.Yes:
        raise RuntimeError("Step 6 was cancelled. Nothing was changed.")

    now = datetime.now(timezone.utc)
    now_iso = now.isoformat()
    temporary_file = participant_file.with_name(
        f".{participant_file.name}.append-{uuid.uuid4().hex}.tmp"
    )
    try:
        if participant_file.exists():
            shutil.copy2(participant_file, temporary_file)
        else:
            with h5py.File(temporary_file, "w") as package_h5:
                package_h5.attrs["format_name"] = (
                    "Anesthesia EEG participant package"
                )
                package_h5.attrs["schema_version"] = "2.0"
                package_h5.attrs["participant_id"] = participant
                package_h5.attrs["package_created_utc"] = now_iso
                package_h5.create_group("recordings")

        # Copy all data and embedded metadata from the individual file.
        with h5py.File(source_file, "r") as source_h5:
            with h5py.File(temporary_file, "r+") as package_h5:
                recording_group = package_h5["recordings"].create_group(
                    recording_group_name
                )

                for attribute_name, attribute_value in source_h5.attrs.items():
                    recording_group.attrs[attribute_name] = attribute_value

                recording_group.attrs["packaged_utc"] = now_iso

                for child_name in ("eeg", "annotations", "review"):
                    source_h5.copy(
                        source_h5[child_name],
                        recording_group,
                        name=child_name,
                    )

                package_h5.attrs["package_updated_utc"] = now_iso
                package_h5.attrs["n_recordings"] = len(
                    package_h5["recordings"]
                )
                package_h5.flush()

        # Read-only validation before replacing the participant package.
        with h5py.File(temporary_file, "r") as validation_h5:
            validation_group = validation_h5["recordings"][
                recording_group_name
            ]

            if "data_zscore" not in validation_group["eeg"]:
                raise RuntimeError(
                    "Validation failed: processed EEG data are missing."
                )

            if validation_group["eeg"]["data_zscore"].ndim != 2:
                raise RuntimeError(
                    "Validation failed: EEG data must be channels x samples."
                )

        # os.replace is atomic when source and destination are on the same drive.
        os.replace(temporary_file, participant_file)

    except Exception:
        if temporary_file.exists():
            temporary_file.unlink()
        raise

    # Confirm the final package is readable before removing the individual file.
    with h5py.File(participant_file, "r") as final_h5:
        final_group = final_h5["recordings"][recording_group_name]
        if "data_zscore" not in final_group["eeg"]:
            raise RuntimeError(
                "Final package validation failed. The individual file was kept."
            )

    source_removed = False
    try:
        source_file.unlink()
        source_removed = True
    except OSError as cleanup_error:
        QMessageBox.warning(
            None,
            "Package updated; cleanup needed",
            (
                "The participant package was updated and validated, but "
                "Windows could not remove the separate reviewed file.\n\n"
                f"Please close programs using this file and delete it manually:\n"
                f"{source_file}\n\nError: {cleanup_error}"
            ),
        )

    QMessageBox.information(
        None,
        "Participant package updated",
        (
            f"Added {recording_group_name} to:\n\n"
            f"{participant_file}\n\n"
            + (
                "The separate reviewed file was removed."
                if source_removed
                else "The separate reviewed file still requires manual deletion."
            )
        ),
    )

    return participant_file, recording_group_name, source_removed


_qt_app()

STEP6_SOURCE_FILE = choose_reviewed_recording_file()

(
    PARTICIPANT_PACKAGE_FILE,
    ADDED_RECORDING_GROUP,
    STEP6_SOURCE_REMOVED,
) = append_reviewed_recording_to_participant_package(
    reviewed_recording_file=STEP6_SOURCE_FILE,
    export_folder=EXPORT_FOLDER,
)

print()
print("Step 6 complete.")
print("Participant package:", PARTICIPANT_PACKAGE_FILE)
print("Added recording group:", ADDED_RECORDING_GROUP)
print("Separate reviewed file removed:", STEP6_SOURCE_REMOVED)


RuntimeError: No reviewed recording was selected.